# RT-DETR — inference pipeline

The pipeline uses 5 saved checkpoints of the RT-DETR configuration with the highest mean mAP across held-out folds.
Main files:
- predict_rtdetr_views.py: Generates predictions using TTA.
- rtdetr_pipeline.py: Handles WBF, TTA view selection, score threshold, temporal processing, official validation evaluation, and Kaggle CSV generation without automatic submission.

First, all 8 TTA views (horizontal flipping and scales of 0.75, 1.0, 1.25, and 1.5) are tested on held-out frames from 56 training videos, using 80 consecutive frames from each video.

In [1]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
PIPELINE_DIR = Path('.').resolve()

In [2]:
chosen = json.loads((PIPELINE_DIR / 'chosen_config.json').read_text())
print('configuration:', chosen['config'])
display(pd.Series(chosen['cv_mAP']).sort_values(ascending=False).round(4).to_frame('mean held-out mAP (5 folds)'))

configuration: no_denoising_640x1152


,mean held-out mAP (5 folds)
no_denoising_640x1152,0.2602
no_denoising,0.2258
lr_backbone_2e-5,0.1724
queries_450,0.1697
recipe_warmup_ema,0.1691
queries_200,0.1663
matching_bbox7_giou3,0.1578


In [3]:
def show_tuning_stage(stage, top=8):
    table = pd.read_csv(PIPELINE_DIR / 'tuning' / f'{stage}.csv').sort_values('mAP', ascending=False)
    fold_columns = [column for column in table.columns if column.startswith('mAP_fold')]
    print(stage, f'({len(table)} settings, top {top})')
    display(table[['key', 'mAP', 'AP50', 'AP75', 'APs'] + fold_columns].head(top).round(4))

In [4]:
for stage in ['r_wbf', 'r_views', 'r_score', 'r_temporal']:
    if (PIPELINE_DIR / 'tuning' / f'{stage}.csv').exists():
        show_tuning_stage(stage)

r_wbf (7 settings, top 8)


,key,mAP,AP50,AP75,APs,mAP_fold0,mAP_fold1,mAP_fold2,mAP_fold3,mAP_fold4
4,all views: NMS 0.5 per view + WBF 0.6,0.3077,0.5626,0.2969,0.1056,0.3387,0.3345,0.3336,0.3181,0.2433
6,all views: NMS 0.5 per view + WBF 0.65,0.3077,0.5610,0.2979,0.1057,0.3387,0.3346,0.3328,0.3182,0.2432
2,all views: NMS 0.5 per view + WBF 0.55,0.3074,0.5636,0.2960,0.1054,0.3382,0.3338,0.3334,0.3181,0.2431
5,all views: WBF 0.65,0.2994,0.5498,0.2884,0.1043,0.3264,0.3267,0.3204,0.3126,0.2349
3,all views: WBF 0.6,0.2977,0.5492,0.2854,0.1044,0.3247,0.3247,0.3197,0.3108,0.2336
1,all views: WBF 0.55,0.2955,0.5479,0.2825,0.1039,0.3232,0.3220,0.3187,0.3058,0.2319
0,single view none@1.0 (trained setting),0.2749,0.5048,0.2609,0.0908,0.3027,0.2933,0.3079,0.2968,0.2158


r_views (8 settings, top 8)


,key,mAP,AP50,AP75,APs,mAP_fold0,mAP_fold1,mAP_fold2,mAP_fold3,mAP_fold4
6,flip+none @ 0.75+1.0+1.25+1.5,0.3077,0.5626,0.2969,0.1056,0.3387,0.3345,0.3336,0.3181,0.2433
5,flip+none @ 1.0+1.25+1.5,0.3060,0.5629,0.2915,0.1030,0.3314,0.3420,0.3354,0.3212,0.2391
3,flip+none @ 1.0+1.5,0.3044,0.5592,0.2895,0.1023,0.3309,0.3386,0.3334,0.3203,0.2366
7,none @ 0.75+1.0+1.25+1.5,0.3041,0.5551,0.2914,0.1057,0.3273,0.3290,0.3341,0.3187,0.2425
2,flip+none @ 1.0+1.25,0.3001,0.5502,0.2871,0.1023,0.3323,0.3283,0.3276,0.3134,0.2367
4,flip+none @ 0.75+1.0+1.25,0.2968,0.5467,0.2855,0.1020,0.3360,0.3159,0.3208,0.3084,0.2342
0,flip+none @ 1.0,0.2821,0.5223,0.2694,0.0951,0.3214,0.3034,0.3082,0.3012,0.2157
1,flip+none @ 0.75+1.0,0.2755,0.5168,0.2583,0.0923,0.3211,0.2872,0.3007,0.2915,0.2163


r_score (4 settings, top 8)


,key,mAP,AP50,AP75,APs,mAP_fold0,mAP_fold1,mAP_fold2,mAP_fold3,mAP_fold4
2,score>=0.05,0.3077,0.5627,0.2969,0.1057,0.3388,0.3345,0.3336,0.3181,0.2433
0,score>=0.01,0.3077,0.5626,0.2969,0.1056,0.3387,0.3345,0.3336,0.3181,0.2433
1,score>=0.02,0.3077,0.5626,0.2969,0.1056,0.3387,0.3345,0.3336,0.3181,0.2433
3,conf max,0.2975,0.5465,0.2853,0.1086,0.3370,0.3264,0.3334,0.3142,0.2284


r_temporal (19 settings, top 8)


,key,mAP,AP50,AP75,APs,mAP_fold0,mAP_fold1,mAP_fold2,mAP_fold3,mAP_fold4
1,Faster R-CNN parameters (frozen),0.3141,0.5739,0.3031,0.1067,0.3513,0.3448,0.3375,0.3311,0.2500
4,s_high 0.2 alpha 0.65 beta 0.6,0.3141,0.5745,0.3026,0.1065,0.3521,0.3446,0.3368,0.3238,0.2509
5,s_high 0.2 alpha 0.65 beta 1.0,0.3139,0.5740,0.3025,0.1062,0.3524,0.3448,0.3368,0.3223,0.2501
11,s_high 0.3 alpha 0.8 beta 0.6,0.3139,0.5736,0.3029,0.1070,0.3523,0.3463,0.3372,0.3312,0.2503
8,s_high 0.3 alpha 0.5 beta 0.6,0.3139,0.5735,0.3029,0.1070,0.3501,0.3432,0.3374,0.3303,0.2493
2,s_high 0.2 alpha 0.5 beta 0.6,0.3139,0.5740,0.3024,0.1070,0.3507,0.3432,0.3368,0.3233,0.2500
6,s_high 0.2 alpha 0.8 beta 0.6,0.3138,0.5741,0.3023,0.1064,0.3529,0.3459,0.3361,0.3240,0.2512
3,s_high 0.2 alpha 0.5 beta 1.0,0.3136,0.5734,0.3023,0.1066,0.3509,0.3434,0.3368,0.3218,0.2493


In [5]:
print(json.dumps(json.loads((PIPELINE_DIR / 'tuning/selected.json').read_text()), indent=1))

{
 "wbf_iou": 0.6,
 "pre_nms": 0.5,
 "views": [
  "none@0.75",
  "h@0.75",
  "none@1.0",
  "h@1.0",
  "none@1.25",
  "h@1.25",
  "none@1.5",
  "h@1.5"
 ],
 "score_min": 0.05,
 "conf_type": "avg",
 "model_temporal": {
  "s_high": 0.3,
  "s_low": 0.01,
  "iou_link": 0.5,
  "max_gap": 2,
  "min_len": 3,
  "alpha": 0.65,
  "beta": 0.6,
  "gamma": 0.75
 },
 "fusion": {
  "weight_fcos": 1.0,
  "iou": 0.65
 },
 "fusion_temporal": {
  "s_high": 0.2,
  "s_low": 0.01,
  "iou_link": 0.5,
  "max_gap": 2,
  "min_len": 3,
  "alpha": 0.65,
  "beta": 0.6,
  "gamma": 0.75
 }
}


## Official validation

In [6]:
official = pd.read_csv(PIPELINE_DIR / 'official_validation_results.csv')
video_columns = [column for column in official.columns if column.startswith('mAP_video_')]
display(official[['key', 'mAP', 'AP50', 'AP75', 'APs', 'APm', 'APl', 'AR']].round(4))

,key,mAP,AP50,AP75,APs,APm,APl,AR
0,Faster R-CNN v2 + temporal (Kaggle 0.3188),0.2841,0.5573,0.2545,0.1560,0.2584,0.4894,0.4996
1,Faster R-CNN v2 + FCOS + temporal (best valida...,0.2928,0.5588,0.2692,0.1660,0.2659,0.5005,0.5113
2,"RT-DETR 5 checkpoints, single view (trained se...",0.2686,0.5134,0.2442,0.1336,0.2443,0.4512,0.5019
3,RT-DETR 5 checkpoints TTA + WBF,0.2882,0.5437,0.2662,0.1425,0.2673,0.4756,0.5259
4,RT-DETR + v2/FCOS ensemble fusion (no temporal),0.2976,0.5586,0.2783,0.1534,0.2767,0.4831,0.5303
5,RT-DETR 5-fold TTA + WBF + temporal (tuned),0.3012,0.5706,0.2766,0.1461,0.2751,0.4986,0.5263
6,RT-DETR + v2/FCOS ensemble fusion + temporal (...,0.3051,0.5771,0.2825,0.1578,0.2834,0.5026,0.5306


In [7]:
per_video = official.set_index('key')[video_columns]
per_video.columns = [column.replace('mAP_video_', '') for column in video_columns]
display(per_video.round(4))

,uav0000086_00000_v,uav0000117_02622_v,uav0000137_00458_v,uav0000182_00000_v,uav0000268_05773_v,uav0000305_00000_v,uav0000339_00001_v
key,,,,,,,
Faster R-CNN v2 + temporal (Kaggle 0.3188),0.3185,0.3279,0.3603,0.2384,0.1612,0.3553,0.3187
Faster R-CNN v2 + FCOS + temporal (best validated so far),0.3034,0.3357,0.3588,0.2415,0.1922,0.3808,0.3404
"RT-DETR 5 checkpoints, single view (trained setting)",0.3602,0.2910,0.3513,0.2072,0.1269,0.4141,0.3078
RT-DETR 5 checkpoints TTA + WBF,0.4092,0.3095,0.3848,0.2186,0.1566,0.4704,0.3124
RT-DETR + v2/FCOS ensemble fusion (no temporal),0.3729,0.3178,0.3848,0.2332,0.1767,0.4506,0.3224
RT-DETR 5-fold TTA + WBF + temporal (tuned),0.4096,0.3287,0.3954,0.2295,0.1723,0.4882,0.3164
RT-DETR + v2/FCOS ensemble fusion + temporal (tuned),0.3952,0.3347,0.3907,0.2422,0.1873,0.4669,0.3300


In [8]:
report = json.loads((PIPELINE_DIR.parents[1] / 'kaggle_submissions' / 'rtdetr_submission_report.json').read_text())
print('decision for the ensemble fusion:', report['decision_ensemble_fusion'])
for name, submission in report['submissions'].items():
    checks = 'all pass' if all(submission['checks'].values()) else submission['checks']
    print(name, '->', Path(submission['submission']).name, '| checks:', checks,
          '| rows', submission['stats']['rows'], '| max boxes/frame', submission['stats']['max_per_image'])

decision for the ensemble fusion: {'mAP': 0.3051202906010878, 'reference_B': 0.29281037428234313, 'gain': 0.012309916318744674, 'videos_improved': 4, 'videos': 7, 'adopt': False}
RT-DETR 5-fold TTA + WBF + temporal (tuned) -> rtdetr_5fold_tta_wbf_temporal_submission.csv | checks: all pass | rows 3811 | max boxes/frame 500
